<a href="https://colab.research.google.com/github/waelrash1/ai-for-business-analytics/blob/main/notebooks/Block%2003/Block3_Lab1_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing and leakage with Python and an AI partner

**MI6261 AI for Business Analytics, Block 3.** Dr Wael Rashwan, School of Business, Maynooth University. MSc in Business Analytics.

The operations manager's question: **when the seller hands a parcel to the carrier, will it arrive late?** Before any model, the data must be prepared: the right features, a fair split, a quality check, numbers on one scale, categories as numbers, and no column that gives the answer away. You will reuse this preparation in Labs 3 and 4.

This lab is **vibe coding with checks**. You do not type the analysis code yourself. In each task you:

1. **Plan**: read the question and decide what one result you need.
2. **Prompt**: ask Gemini or Claude for **one small code cell**, using the five headings (Context, Task, Constraints, Format, Check).
3. **Read** the code before you run it: is anything fitted on the test set? Is a leaking column used?
4. **Run** it.
5. **Verify** one number in Excel or by a second method.
6. **Explain** the result in your own words, for a manager.

Every task gives you the full prompt, written with the five headings. Copy it into Gemini or Claude, then study why it works: this lab is practice in reading, using and adapting good prompts.

**Rules.** Use only what is known at the moment of the decision. Split before you fit. Learn every preparation step from the training rows only.

**Data.** 5,000 real delivered orders from the Olist Brazilian E-Commerce Public Dataset (Kaggle, CC BY-NC-SA 4.0), cleaned with the rules of the Block 2 data cleaning lab (one row per order, real dates, delivered orders only, consistent state codes), with product weight, distance, seller state, promised days and handling days joined from the other Olist tables.

- **Orders**: https://raw.githubusercontent.com/waelrash1/ai-for-business-analytics/refs/heads/main/olist_model_orders.csv

**Extra practice.** The cleaning steps themselves (tasks 1 to 10) are in the Block 2 data cleaning notebook.

**AI use.** You may use any AI to write code. You must read, check and explain every cell, and keep a prompt log (end of the notebook).

---
## Task 1: Load the orders, choose the features, add the target

**Question.** When the seller hands a parcel to the carrier, what do we know, and what do we want to predict?

**Plan.** One cell that produces: `n_rows`, `n_late`, `n_features`.

**A weak prompt:** *"Load the data and get it ready for a model."* Ready for which question, at which moment? Name the target and the moment of decision first; they decide which columns are allowed.

In [ ]:
import pandas as pd
orders_url = "https://raw.githubusercontent.com/waelrash1/ai-for-business-analytics/refs/heads/main/olist_model_orders.csv"

**The prompt.** I use this one on screen. Copy it into Gemini or Claude.

```text
Context: Colab. `orders_url`: 5,000 cleaned, delivered Olist orders, one row per order.
Task: Load as `orders` (order_id as text). Add late: 1 if days_late above 0. X = the nine columns known at handover (handling_days, promised_days, distance_km, weight_g, freight, n_items, purchase_month, both states); y = late.
Constraints: pandas only. Never delivery_days, days_late or review_score in X.
Format: One cell; print the shapes and the late share.
Check: Store `n_rows`, `n_late` and `n_features`.
```

**Why this prompt is better.** Compare the weak prompt, *"Load the data and get it ready for a model."* Its problem: Ready for which question, at which moment? Name the target and the moment of decision first; they decide which columns are allowed. A strong AI may still get this right from the weak prompt, but then the AI made the decision, not you. The **Task** and **Constraints** lines state your decision, so the result is yours and you can check it. Try both prompts if you have time and compare the code. The checklist under the next heading is how you confirm the code follows your decision.

**Paste the AI's code below.** Before you run it, check that it:
- late uses > 0: on the promised day is on time
- No delivery_days, days_late or review_score in X
- Nothing is dropped or changed

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Verify in Excel.**

- Late orders in olist_model_orders.csv: `=COUNTIF(N2:N5001,">0")`

In [ ]:
excel_value = None   # type what Excel shows
print(n_late == excel_value)   # True means Python and Excel agree

**Explain.** delivery_days would make the model far better. Why is it still not allowed?

In [ ]:
my_explanation = """

"""

---
## Task 2: Split before you fit anything

**Question.** How do we keep a fair test for later?

**Plan.** One cell that produces: `n_train`, `n_late_test`.

**A weak prompt:** *"Scale the data, then split it."* Scaling first lets the test rows shape the training data: a quiet leak. Split first, and keep the late share the same in both parts.

**The prompt.** I use this one on screen. Copy it into Gemini or Claude.

```text
Context: `X` and `y` from task 1; only about 7 orders in 100 are late.
Task: Split 75/25 into X_train, X_test, y_train, y_test.
Constraints: train_test_split, random_state=42, stratify=y. Nothing fitted yet.
Format: One cell; the sizes and the late share in each part.
Check: Store `n_train` and `n_late_test`.
```

**Why this prompt is better.** Compare the weak prompt, *"Scale the data, then split it."* Its problem: Scaling first lets the test rows shape the training data: a quiet leak. Split first, and keep the late share the same in both parts. A strong AI may still get this right from the weak prompt, but then the AI made the decision, not you. The **Task** and **Constraints** lines state your decision, so the result is yours and you can check it. Try both prompts if you have time and compare the code. The checklist under the next heading is how you confirm the code follows your decision.

**Paste the AI's code below.** Before you run it, check that it:
- stratify=y
- random_state=42
- The split comes before any scaling or encoding

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: train and test together are all 5,000 orders

In [ ]:
print(n_train + len(X_test) == len(orders))   # train and test are all the orders

**Explain.** Why does stratify matter more here than it would if half the orders were late?

In [ ]:
my_explanation = """

"""

---
## Task 3: A quality report and the outliers, from the training rows

**Question.** Before we scale anything: is anything missing, how many values does each column take, and are the very heavy parcels errors or real?

**Plan.** One cell that produces: `n_states`, `upper_fence`, `n_heavy_train`.

**A weak prompt:** *"Remove the outliers."* Which ones, by which rule, and are they wrong or just rare? A 100 kg parcel is a real order. Measure first, on the training rows, then decide.

**The prompt, written for you.** Copy it into Gemini or Claude.

```text
Context: `X_train` from task 2; `nums` and `cats` from task 1.
Task: A quality table for X_train: missing % and cardinality (distinct values) per column. Then the box plot upper fence for weight_g (Q3 + 1.5 IQR) from the training rows, and how many training orders lie above it.
Constraints: pandas only; training rows only; do not drop or change any rows.
Format: One cell; print the table and two labelled numbers.
Check: Store `n_states` (distinct customer states in train), `upper_fence` (1 decimal) and `n_heavy_train`.
```

**Why this prompt is better.** Compare the weak prompt, *"Remove the outliers."* Its problem: Which ones, by which rule, and are they wrong or just rare? A 100 kg parcel is a real order. Measure first, on the training rows, then decide. A strong AI may still get this right from the weak prompt, but then the AI made the decision, not you. The **Task** and **Constraints** lines state your decision, so the result is yours and you can check it. Try both prompts if you have time and compare the code. The checklist under the next heading is how you confirm the code follows your decision.

**Paste the AI's code below.** Before you run it, check that it:
- Quartiles from X_train, not from all orders
- Nothing is dropped or clipped
- Cardinality counts distinct values, not rows

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: describe() gives the same quartiles, so the same fence

In [ ]:
print((lambda d: round(d['75%'] + 1.5 * (d['75%'] - d['25%']), 1))(X_train['weight_g'].describe()) == upper_fence)   # describe() gives the same fence

**Explain.** Are the heavy parcels above the fence errors or valid orders? Say what you would do with them for a logistic regression, and for a tree model.

In [ ]:
my_explanation = """

"""

---
## Task 4: Scale the numbers, learning from train only

**Question.** Distance runs to thousands of kilometres and items to a handful. How do we put them on one scale without peeking at the test set?

**Plan.** One cell that produces: `train_mean_scaled`, `test_mean_scaled`.

**A weak prompt:** *"Standardise all the numeric columns."* On which rows? A scaler fitted on all 5,000 orders has seen the test set. Fit on train, then apply the same scaler to test.

**The prompt, written for you.** Copy it into Gemini or Claude.

```text
Context: `X_train`, `X_test` from task 2; `nums` lists the seven numeric features.
Task: StandardScaler fitted on X_train[nums] only; transform both. Mean of scaled distance_km in train and in test.
Constraints: fit on train only; transform test with the same scaler.
Format: One cell; two labelled numbers.
Check: Store `train_mean_scaled` and `test_mean_scaled` (3 decimals).
```

**Why this prompt is better.** Compare the weak prompt, *"Standardise all the numeric columns."* Its problem: On which rows? A scaler fitted on all 5,000 orders has seen the test set. Fit on train, then apply the same scaler to test. A strong AI may still get this right from the weak prompt, but then the AI made the decision, not you. The **Task** and **Constraints** lines state your decision, so the result is yours and you can check it. Try both prompts if you have time and compare the code. The checklist under the next heading is how you confirm the code follows your decision.

**Paste the AI's code below.** Before you run it, check that it:
- fit on X_train only
- transform, not fit_transform, on X_test
- Only the numeric columns are scaled

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: the scaler's mean for distance_km is the training mean, not the mean of all orders

In [ ]:
print(round(scaler.mean_[nums.index('distance_km')], 6) == round(X_train['distance_km'].mean(), 6))   # the scaler learned the TRAINING mean

**Explain.** The training mean is exactly 0 and the test mean is not. Is that a bug? What would it mean if the test mean were also exactly 0?

In [ ]:
my_explanation = """

"""

---
## Task 5: One pipeline for every step

**Question.** Can the same preparation run on new orders next month, with no step forgotten?

**Plan.** One cell that produces: `n_prepared`.

**A weak prompt:** *"Encode the states."* Encoded where, and what happens with a state the model has never seen? Put scaling and encoding in one ColumnTransformer, fitted on train.

**The prompt, written for you.** Copy it into Gemini or Claude.

```text
Context: `X_train`, `X_test`, `nums`, `cats` (the two state columns).
Task: ColumnTransformer: StandardScaler on nums, OneHotEncoder on cats. Fit on train; transform both.
Constraints: OneHotEncoder(handle_unknown="ignore"); fit on train only.
Format: One cell; print the number of columns after preparation.
Check: Store `prep` and `n_prepared` (columns after preparation).
```

**Why this prompt is better.** Compare the weak prompt, *"Encode the states."* Its problem: Encoded where, and what happens with a state the model has never seen? Put scaling and encoding in one ColumnTransformer, fitted on train. A strong AI may still get this right from the weak prompt, but then the AI made the decision, not you. The **Task** and **Constraints** lines state your decision, so the result is yours and you can check it. Try both prompts if you have time and compare the code. The checklist under the next heading is how you confirm the code follows your decision.

**Paste the AI's code below.** Before you run it, check that it:
- One ColumnTransformer, fitted on train
- handle_unknown='ignore' for states not seen in training
- transform only on test

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: train and test have the same columns after preparation

In [ ]:
print(X_train_ready.shape[1] == X_test_ready.shape[1])   # train and test have the same columns

**Explain.** Seven numbers became many more columns. Where did the extra columns come from, and what happens to an order from a state that is not in the training data?

In [ ]:
my_explanation = """

"""

---
## Task 6: Find the leak

**Question.** A colleague's model is far better than ours. They added one more column. Should we copy them?

**Plan.** One cell that produces: `auc_clean`, `auc_leak`.

**A weak prompt:** *"Add more columns to improve the model."* A score that jumps is a warning, not a win. Compare the model with and without the extra column, and ask when that column is known.

**The prompt, written for you.** Copy it into Gemini or Claude.

```text
Context: `prep`, the train and test sets, `orders`. The colleague added delivery_days.
Task: Two pipelines, prep then LogisticRegression: one on our features, one with delivery_days added as a number. ROC AUC of each on test.
Constraints: max_iter=1000; the same split (use the index of X_train and X_test).
Format: One cell; two labelled AUCs.
Check: Store `auc_clean` and `auc_leak` (3 decimals).
```

**Why this prompt is better.** Compare the weak prompt, *"Add more columns to improve the model."* Its problem: A score that jumps is a warning, not a win. Compare the model with and without the extra column, and ask when that column is known. A strong AI may still get this right from the weak prompt, but then the AI made the decision, not you. The **Task** and **Constraints** lines state your decision, so the result is yours and you can check it. Try both prompts if you have time and compare the code. The checklist under the next heading is how you confirm the code follows your decision.

**Paste the AI's code below.** Before you run it, check that it:
- The same train and test rows for both models
- delivery_days is the only difference
- AUC from probabilities, on test

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: the model with the leaking column scores higher

In [ ]:
print(auc_leak > auc_clean)   # the leaking column inflates the score

**Explain.** Write two sentences for the colleague: why the higher score is not real, and what question to ask of any column before using it.

In [ ]:
my_explanation = """

"""

---
## Your preparation log (about 150 words)

**Features.** Which columns you kept and why each is known at handover.

**Split.** How you split and why.

**Quality.** What the quality report showed, and what you decided about the heavy parcels.

**Preparation.** What the pipeline does to numbers and to categories, and what it learned from the training rows.

**Leakage.** The column you rejected, and the question you asked of it.

**AI use.** One thing in the AI's code you had to correct, or would have had to.

---
## Your prompt log

Vibe coding is only safe when you check. Fill in three rows before you leave.

| Task | The prompt that worked (short) | What the AI got wrong, or could have | How you caught it |
|---|---|---|---|
| | | | |
| | | | |
| | | | |